In [1]:
!pip install lxml

  Using cached lxml-6.0.2-cp39-cp39-macosx_10_9_universal2.whl.metadata (3.6 kB)
Using cached lxml-6.0.2-cp39-cp39-macosx_10_9_universal2.whl (8.6 MB)


In [ ]:
from lxml import etree
import re

# =========================
# PARAMÈTRES
# =========================
ARK = "bpt6k1051278h"
START_F = 403

BASE_URL = f"https://gallica.bnf.fr/ark:/12148/{ARK}/f{{}}.item"

NSMAP = {
    None: "http://www.tei-c.org/ns/1.0",
    "xi": "http://www.w3.org/2001/XInclude"
}

# =========================
# PARSER (IMPORTANT)
# =========================
parser = etree.XMLParser(
    remove_blank_text=True,
    strip_cdata=False
)

tree = etree.parse("../../corpus/Peinture/Piles_CoursPeinture.xml", parser)
root = tree.getroot()

# =========================
# EXTRACTION fXXX
# =========================
def extract_f_number(facs_value):
    match = re.search(r'/f(\d+)\.item', facs_value)
    if match:
        return int(match.group(1))
    return None

# =========================
# TRAITEMENT DES <pb>
# =========================
current_f = START_F

for pb in root.xpath('//tei:pb', namespaces={'tei': NSMAP[None]}):
    facs = pb.get('facs')

    if facs:
        extracted = extract_f_number(facs)
        if extracted:
            current_f = extracted
    else:
        current_f += 1
        pb.set('facs', BASE_URL.format(current_f))

# =========================
# FIX xi:include (éviter ns1)
# =========================
for el in root.xpath('//*[local-name()="include"]'):
    el.tag = "{http://www.w3.org/2001/XInclude}include"

# =========================
# ÉCRITURE SANS ESPACES EN TROP
# =========================
tree.write(
    "output.xml",
    encoding="UTF-8",
    xml_declaration=True,
    pretty_print=True
)

print("✔ XML propre, namespaces conservés, pas de ns1")

✔ XML propre, namespaces conservés, pas de ns1
